# Create project and initiziale it

First, we create a folder for our work and initialize it with `uv` to manage our packages.  
The `--no-workspace` flag ensures our project remains clean and independent.
- `mkdir proj_zoomcamp_02_vector_search`
- `cd proj_zoomcamp_02_vector_search`
- `uv init --no-workspace`

We need specific libraries to run the ONNX model, process text, and build our search index. We also add `jupyter` and `huggingface-hub` as development tools to help us experiment.

- `uv add onnxruntime tokenizers numpy tqdm minsearch gitsource`
- `uv add --dev huggingface-hub jupyter`

We need two scripts from the course repository to handle model downloading (`download.py`) and vector embedding (`embedder.py`). We use a shortcut (`PREFIX`) to download them cleanly.


- `PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed`

- `curl -O $PREFIX/download.py`
- `curl -O $PREFIX/embedder.py`

Now that we have the `download.py` script, we run it to fetch the actual machine learning model (by default, Xenova/all-MiniLM-L6-v2) needed to turn text into vectors.

- `un run python download.py`

# Homework

## Q1. Embedding a query
Embed the following query:
>- "How does approximate nearest neighbor search work?"

**The embedder returns a vector of 384 numbers. What's the first value (`v[0]`)?**
- -0.31
- **-0.02**
- -0.12
- -0.44



In [4]:
# Import the embedder
from embedder import Embedder

# Initialize the embedder
embedder = Embedder()
# Embed the query
query_vector = embedder.encode("How does approximate nearest neighbor search work?")

# Print the query vector
print(query_vector[0])

-0.020582036807885073


## Loading the data
We pull the **lesson pages** from the course repository, the same way as in homework 1. We pin to commit `8c1834d` so everyone works with the same data.  

This will return a `list[dict]` where each item is a lesson (dictionary) with a filename and actual content, and there are 72 pages.

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [6]:
documents[:5]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

## Q2. Cosine similarity
The embedder returns normalized vectors (length/magnitude of 1), so the dot product between two of them is their cosine similarity.

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

- 0.07
- **0.37**
- 0.68
- 0.92

In [7]:
for doc_dict in documents:
    if doc_dict['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md':
        lesson = doc_dict

In [8]:
lesson_vector = embedder.encode(lesson['content'])
print(type(lesson_vector))
query_vector.dot(lesson_vector)

<class 'numpy.ndarray'>


np.float64(0.361070280302606)

## Q3. Chunking and search by hand
Score the Q1 query against the documents (content chunks) and answer:  

***Which file does the highest-scoring chunk belong to (its filename)?***

- 02-vector-search/lessons/03-embeddings-dataset.md
- 02-vector-search/lessons/06-rag-vector.md
- **02-vector-search/lessons/07-sqlitesearch-vector.md**
- 02-vector-search/lessons/09-onnx-embedder.md

A full page covers several topics, therefore when we embed a large document/page into one single vector, that vector has to represent everything in the page.  

If the page talks about several different topics, the embedding becomes a kind of “average meaning” of all those topics. That can make it less strongly similar to a specific query.  

For example, suppose a page contains:
- SQLite setup
- Vector search
- Approximate nearest neighbors
- Index configuration
- Performance tuning
If your query is:
> "How does approximate nearest neighbor search work?"  

Only part of the page is really about that. But the page embedding also includes other topics, so the similarity score may be lower. The relevant signal is diluted by unrelated or less relevant content.

In **vector search**, this is why we often **split documents into smaller chunks** before embedding them. A focused chunk about “*approximate nearest neighbor search*” will usually match the query better than a full page containing many topics.

In [9]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
chunks[:2]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

The method of the embedding model `encode_batch` is to encode many texts at once as a list of strings.  

For example:
```python
vectors = embedder.encode_batch([
    "First text",
    "Second text",
    "Third text",
])
```

will return a matrix of vector whose shape is (3,384)

In [11]:
lesson_content_chunks = [chunk['content'] for chunk in chunks]

# This will return a matrix (2D numpy array) of vectors
matrix_content_chunks = embedder.encode_batch(lesson_content_chunks)

In [12]:
# We compute the similarity between the user query vector (from Q1) and the lesson chunks vectors
scores_vector = matrix_content_chunks.dot(query_vector)

import numpy as np
# Find the index of the max similarity (lesson chunk most relevant to the user query)
max_similarity_index = np.argmax(scores_vector)

# Get the lesson filename of the most relevant lesson chunk
lesson_filename = chunks[max_similarity_index]['filename']

# Print the lesson filename
print(lesson_filename)



02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4. Vector search with minsearch
We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

>***What metric do we use to evaluate a search engine?***

Which file is the filename of the first result?

- 02-vector-search/lessons/04-vector-search.md
- **04-evaluation/lessons/05-search-metrics.md**
- 04-evaluation/lessons/13-llm-as-judge.md
- 05-monitoring/lessons/04-metrics.md

### 1. Build the `minsearch` index

The first step to enable a vector search is to build a index. The index, in this case, is a VectorSearch object. We create the index using the `fit()` method which takes as input two arguments: 
1. ***A 2D arrays of vectors***: each row is the embedding of each chunk, not the whole lesson page.  
    **Example:**
    > `07-sqlitesearch-vector.md` gets split into multiple overlapping chunks first, and each of those chunks becomes one row in the embedded 2D Matrix. 
2. ***A payload***: A list of dictionaries that contains the plain (non-embedding) text of each of the chunks in the index.

>**The whole lesson page itself is never embedded directly, only its chunks are.**

Since each lesson page contains several paragraphs and topics, previously we chunked the documents (all lessons) into smaller chunks (chunk size was 2000 characters in steps of 1000 characters. Yes, there is an overlap between each chunk to avoid splitting one concept in two different chunks.). 

Therefore, the 2D array of vectors we fit into the index is the embedding of the chunked version of the documents and the payload is the plain chunked version of the documents


In [13]:
from minsearch import VectorSearch

# To look for the class signature we can run help(VectorSearch)
# help(VectorSearch)

In [14]:
# Create an empty vector search index
vindex = VectorSearch()

# Fit the index with the 2D array of vectors and the payload
vindex.fit(matrix_content_chunks, chunks)

### 2. Search the relevant lesson for the user query

Assume we now recevie a user query:  
> *"What metric do we use to evaluate a search engine?"*  

to sematically search in the index for a lesson's chunk that contain the information to answer this user question, we first need to **embed the user query** before launching the actual search.


In [15]:
user_query = "What metric do we use to evaluate a search engine?"
v_user_query = embedder.encode(user_query)
search_results = vindex.search(v_user_query, num_results=3)

In [16]:
# Print the filename of the first result to answer Q4
print(search_results[0]['filename'])

04-evaluation/lessons/05-search-metrics.md


## Q5. Text search vs Vector search  
Vector search matches by meaning, keyword search (aka text search) matches by exact words.

We want to compare them to better understand them.  

Index the same chunks with `Index` from `minsearch`. Use the key `content` as a text field (searchable text).  

Run both searches for this query:

> *How do I store vectors in PostgreSQL?*  

Take the top 5 results from each method.  
Which file shows up in the vector results but not in the text results?

- *02-vector-search/lessons/01-intro.md*
- *02-vector-search/lessons/02-embeddings.md*
- ***02-vector-search/lessons/08-pgvector.md***
- *03-orchestration/lessons/05-rag.md*



In [17]:
user_query = "How do I store vectors in PostgreSQL?"

### Text Search

In [18]:
from minsearch import Index

# The instantiation of the index takes a list of the field we want to search by (content)
index = Index(["content"])

# Fit the index by passing the list of dictionaries (chunks)
index.fit(chunks)

In [19]:
# Perform the text search and print the results
text_search_results = index.search(user_query, num_results=5)


In [20]:
# To answer the question, we need to print the filename of the results for the text search
for result_dict in text_search_results:
    print(result_dict['filename'])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


### Vector Search


In [ ]:
# Embed the user query for the vector search - We will the vector index from before (vindex)
v_user_query = embedder.encode(user_query)

In [22]:
# We will use the already fit index (vindex) for vector search
vector_search_results = vindex.search(v_user_query, num_results=5)

# To answer the question, we need to print the filename of the results for the vector search
for result_dict in vector_search_results:
    print(result_dict['filename'])
    

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


## Q6. Hybrid search
Both vector and text search have their strengths and weaknesses:  

- **Vector search** matches by meaning, so it finds relevant pages even when they use words different from the query. But it can miss exact terms like names, codes, or rare keywords.  
- **Text search** is the opposite: it nails exact words but misses paraphrases and synonyms.  

> We don't have to pick one or the other: we can use both and merge their results.  
> This approach is called ***"hybrid search"***.  

Each search produces its own ranked list for the most relevant *documents*, so we need a way to combine them into one.  

#### Hybrid Search with Reciprocal Rank Fusion (RRF)

We will use ***Reciprocal Rank Fusion (RRF)***. It ignores the raw scores from each method (for vector search the raw scores are the cosine similarity, and for text search they are TF-IDF scores based on word frequency), which have different unit scales and aren't directly comparable (like comparing mg and km...not possible).  

Instead, **RRF looks only at the position of each document in each list**.  

Every document scores is based on on its position (starting at 0) in each list.

>**Example**:  
> In a *document* whose similariy or TF-IDF score was the highest in their list, would be the first in that list having position `0`. 

For each document in the ranked lists (one list for vector search and one for the text search in this case) we sum the documents scores using a constant `k=60` like so:

`RRF(document) = sum( 1 / (k + position(document)) )`  

**In other words:** we go over the first list (vector search results) and for each *document* we add `1/(60 + document_position)` and repeat for the second list (text search results).  

>**An Important Note:**
>A document that **ranks well in both lists** usually **wins over one that is strong in only one list**.

#### How does `k` controls the rank?
The constant `k` controls how much the exact rank matters. A larger `k` reduces the gap between positions, so the difference between rank `0` and rank `5` matters less.  
A smaller `k` increses the rank gap, so top positions (0,1,2...) matter much more than lower positions.  

The value `k=60` comes from the original RRF paper and is the usual default. We rarely need to tune it. 
> **We lower it when only the top results matter**.  
> **We raise it to reward documents that appear across many lists, even when they never quite reach the top.**

A document that ranks well in both lists ends up higher than one that's only strong in a single list.

In [23]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        # Enumerate returns a tuple (index, value) for each item in the iterable
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Now we run the user query with **vector** and **text search** and fuse the results with rrf.  

User query: ***"How do I give the model access to tools?"***  

**Which file is ranked first after RRF?**

- 01-agentic-rag/lessons/01-intro.md
- **01-agentic-rag/lessons/13-function-calling.md**
- 01-agentic-rag/lessons/14-agentic-loop.md
- 01-agentic-rag/lessons/16-other-frameworks.md

In [25]:
user_query = "How do I give the model access to tools?"

In [26]:
# Text Search
text_search_results = index.search(user_query, num_results=5)

In [27]:
# Vector Search

# 1. Embed the user query
v_user_query = embedder.encode(user_query)
# 2. Run the vector search
vector_search_results = vindex.search(v_user_query, num_results=5)

In [28]:
results = rrf([text_search_results, vector_search_results])

In [32]:
print(f"The best result for hybrid search was the lesson:\n{results[0]['filename']}")

The best result for hybrid search was the lesson:
01-agentic-rag/lessons/13-function-calling.md
